In [ ]:
import os
import bpy
import math
import numpy as np
import open3d as o3d
import plotly.graph_objects as go

from pathlib import Path

In [ ]:
bpy.ops.wm.open_mainfile(filepath="/home/addai/Blender/master_chief.blend")

In [ ]:
scene = bpy.context.scene

resolution_x = scene.render.resolution_x
resolution_y = scene.render.resolution_y
scale = scene.render.resolution_percentage / 100
width = resolution_x * scale
height = resolution_y * scale

In [ ]:
width

In [ ]:
camera = bpy.data.objects["Camera"]
cam_data = camera.data

In [ ]:
sensor_width_mm = cam_data.sensor_width  # in mm (typically 36 for full-frame)
focal_length_mm = cam_data.lens  # in mm

focal_x = (focal_length_mm / sensor_width_mm) * width
focal_y = (
    focal_x * (height / width)
    if cam_data.sensor_fit != "VERTICAL"
    else (focal_length_mm / cam_data.sensor_height) * height
)

In [ ]:
target_height, target_width = 1024, 1024

camera_angle_x = 0.9500215649604797
focal_length = 0.5 * width / np.tan(0.5 * camera_angle_x)
focal_length = focal_length * target_width / width
fx = focal_length
fy = focal_length
cx = target_width / 2.0
cy = target_height / 2.0

In [ ]:
print(cam_data.sensor_width)
print(cam_data.sensor_height)

In [ ]:
scene.render.resolution_x

In [ ]:
def blender_mesh_to_open3d(obj_name: str) -> o3d.geometry.TriangleMesh:
    # Get the Blender object
    obj = bpy.data.objects[obj_name]
    mesh = obj.to_mesh()
    mesh.calc_loop_triangles()

    # Transform vertices to world space
    vertices = np.array([obj.matrix_world @ v.co for v in mesh.vertices])

    # Collect triangle indices
    triangles = []
    for tri in mesh.loop_triangles:
        triangles.append([tri.vertices[0], tri.vertices[1], tri.vertices[2]])
    triangles = np.array(triangles)

    # Create Open3D triangle mesh
    o3d_mesh = o3d.geometry.TriangleMesh()
    o3d_mesh.vertices = o3d.utility.Vector3dVector(vertices)
    o3d_mesh.triangles = o3d.utility.Vector3iVector(triangles)
    o3d_mesh.compute_vertex_normals()

    return o3d_mesh

In [ ]:
body_mesh = blender_mesh_to_open3d("Body")

In [ ]:
body_points = body_mesh.sample_points_poisson_disk(10000)
body_points = np.asarray(body_points.points)
body_colors = 120 * np.ones_like(body_points)

In [ ]:
from shadow_splat.util.general import generate_ply_from_points

generate_ply_from_points(body_points, body_colors, "body.ply")

In [ ]:
plane = blender_mesh_to_open3d("Plane")
plane_points = plane.sample_points_poisson_disk(10000)
plane_points = np.asarray(plane_points.points)
plane_colors = 255 * np.ones_like(plane_points)

generate_ply_from_points(plane_points, plane_colors, "plane.ply")

In [ ]:
np.concatenate([plane_points, body_points], axis=0)

In [ ]:
all_points = np.concatenate([plane_points, body_points], axis=0)
all_colors = np.concatenate([plane_colors, body_colors])

generate_ply_from_points(all_points, all_colors, "all_points.ply")

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=all_points[:, 0],
        y=all_points[:, 1],
        z=all_points[:, 2],
        mode="markers",
        marker=dict(size=2, color="blue"),
    )
)
fig.show()

In [ ]:
body = bpy.data.objects["Body"]
mesh = body.to_mesh()
mesh.calc_loop_triangles()

In [ ]:
# List all mesh objects
for obj in bpy.data.objects:
    if obj.type == "MESH":
        print(f"Mesh: {obj.name}, Vertices: {len(obj.data.vertices)}")

In [ ]:
mesh.loop_triangles

In [ ]:
bpy.context.scene.render.engine = "CYCLES"
bpy.context.scene.render.resolution_x = 1024
bpy.context.scene.render.resolution_y = 1024
bpy.context.scene.cycles.samples = 128
# bpy.context.scene.render.image_settings.color_mode = "BW"
# bpy.context.scene.render.image_settings.color_depth = "8"
# bpy.context.scene.render.image_settings.color_depth = "8"

In [ ]:
bpy.data.objects["Camera"]

In [ ]:
from mathutils import Matrix


def set_camera_pose(cam, location, target):
    direction = (target[0] - location[0], target[1] - location[1], target[2] - location[2])
    direction = np.array(direction)  # This is -z
    direction = direction / np.linalg.norm(direction)

    up = np.array([0, 0, 1])

    right = np.cross(up, -direction)
    up = np.cross(-direction, right)

    matrix_world = np.eye(4)
    rotation = np.stack([right, up, -direction], axis=-1)
    translation = location
    matrix_world[:3, :3] = rotation
    matrix_world[:3, -1] = translation
    cam.matrix_world = Matrix(matrix_world)

    return matrix_world

In [ ]:
# Set output path and format
bpy.context.scene.render.image_settings.file_format = "PNG"  # or 'JPEG', etc.
bpy.context.scene.render.filepath = "rendered_image.png"

# Set camera position based on azimuth and elevation


# # Set camera parameters
# azimuth = 45  # degrees
# elevation = 30  # degrees
# distance = 5  # distance from origin

# # Convert spherical to Cartesian coordinates
# x = distance * math.cos(math.radians(elevation)) * math.cos(math.radians(azimuth))
# y = distance * math.cos(math.radians(elevation)) * math.sin(math.radians(azimuth))
# z = distance * math.sin(math.radians(elevation))

# # Position camera
# camera = bpy.data.objects["Camera"]
# camera.location = (x, y, z)

# # Point camera at origin
# direction = camera.location
# rot_quat = direction.to_track_quat('-Z', 'Y')
# camera.rotation_euler = rot_quat.to_euler()

# # Set as active camera
# bpy.context.scene.camera = camera

camera = bpy.data.objects["Camera"]


# Render the image and save it to disk
bpy.ops.render.render(write_still=True)